# 03 · Fine-Tuning with LoRA (run on Google Colab T4)

**Where to run**: [Google Colab](https://colab.research.google.com) → Runtime → Change runtime type → **T4 GPU**

**Estimated training time**: 10–20 minutes on a free T4

By the end of this notebook you will have:
- Understood *why* LoRA is used instead of full fine-tuning
- Trained a LoRA adapter on SmolLM2-135M for 1 epoch
- Saved your adapter to `outputs/adapter/` (download and use in notebooks 04 & 05)

## Setup

In [ ]:
!pip install -q transformers datasets peft trl accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. HF Login

Needed to push your adapter to the Hub in notebook 05. Skip if you just want to train locally.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # enter your HF token with write access

## 2. Load the Base Model & Tokenizer

In [ ]:
MODEL_ID = "HuggingFaceTB/SmolLM2-135M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# SmolLM2 tokenizer doesn't set a pad token — use EOS
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype)

print(f"Loaded {sum(p.numel() for p in model.parameters())/1e6:.0f}M params")
print(f"dtype: {dtype}")

## 3. Why LoRA?

Full fine-tuning updates **all** 135M parameters — large gradient + optimizer state per param.

**LoRA** (Low-Rank Adaptation) instead:
1. Freezes the original weights
2. Adds small **rank-r** matrices `A` and `B` next to key weight matrices
3. Only trains `A` and `B` — a tiny fraction of params

For a weight matrix `W` of shape `[d, k]`, LoRA adds `W + α/r * BA` where `B ∈ R^{d×r}`, `A ∈ R^{r×k}`, `r << d, k`.

**Result**: ~0.5% trainable params, much smaller checkpoint (~5 MB adapter vs 270 MB full).

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,            # rank — controls adapter expressiveness (4–16 is common)
    lora_alpha=16,  # scaling: effective lr = alpha / r. alpha=2r is a safe default
    lora_dropout=0.05,
    # Which weight matrices to adapt — query and value projections in attention
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params:  {trainable:,}  ({trainable/total*100:.2f}% of {total/1e6:.0f}M total)")

In [ ]:
# Inspect which layers have LoRA adapters
print("Layers with LoRA adapters:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name:60s}  shape={list(param.shape)}")

## 4. Load & Prepare the Dataset

In [ ]:
DATASET_ID = "HuggingFaceTB/smoltalk"
DATASET_CONFIG = "everyday-conversations"
N_TRAIN = 2000
N_EVAL = 200

stream = load_dataset(DATASET_ID, DATASET_CONFIG, split="train", streaming=True)
raw_samples = list(stream.take(N_TRAIN + N_EVAL))

from datasets import Dataset
train_ds = Dataset.from_list(raw_samples[:N_TRAIN])
eval_ds = Dataset.from_list(raw_samples[N_TRAIN:])

print(f"Train: {len(train_ds)} samples")
print(f"Eval:  {len(eval_ds)} samples")
print(f"Columns: {train_ds.column_names}")

## 5. Configure SFTTrainer

**`SFTTrainer`** (from TRL — Transformer Reinforcement Learning library) handles:
- Applying the chat template to `messages` column automatically
- **Prompt loss masking** — loss is only computed on assistant turns
- **Sequence packing** — concatenates short samples to fill `max_seq_length` efficiently
- Standard training loop with logging

In [ ]:
OUTPUT_DIR = "outputs/adapter"

sft_config = SFTConfig(
    # --- Output ---
    output_dir=OUTPUT_DIR,

    # --- Training duration ---
    num_train_epochs=1,
    max_steps=-1,  # -1 means use num_train_epochs

    # --- Batch & gradient accumulation ---
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,  # effective batch = 4 * 2 = 8

    # --- Sequence length ---
    max_seq_length=1024,

    # --- Learning rate schedule ---
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # --- Optimizer ---
    optim="adamw_torch",
    weight_decay=0.01,

    # --- Logging & eval ---
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,

    # --- Precision ---
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),

    # --- SFT-specific ---
    # dataset_text_field is NOT set — we pass `messages` and SFTTrainer applies the chat template
    packing=True,  # pack multiple short samples into one sequence for efficiency
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

print("Trainer configured.")
print(f"Steps per epoch: ~{len(trainer.train_dataset) // sft_config.per_device_train_batch_size}")

## 6. Train!

Watch the **train/loss** go down. A starting value of ~2.5 dropping to ~1.5 over 1 epoch is normal for 135M.

In [ ]:
train_result = trainer.train()
print()
print("=== Training complete ===")
print(f"Total steps:   {train_result.global_step}")
print(f"Train loss:    {train_result.training_loss:.4f}")
print(f"Runtime:       {train_result.metrics['train_runtime']:.0f}s ({train_result.metrics['train_runtime']/60:.1f} min)")
print(f"Samples/sec:   {train_result.metrics['train_samples_per_second']:.1f}")

## 7. Save the Adapter

The adapter checkpoint is only ~5 MB — just the LoRA `A` and `B` matrices.

In [ ]:
# Save only the LoRA adapter (not the full base model)
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import os
total_bytes = sum(
    os.path.getsize(os.path.join(root, f))
    for root, _, files in os.walk(OUTPUT_DIR)
    for f in files
)
print(f"Adapter saved to: {OUTPUT_DIR}/")
print(f"Total size: {total_bytes/1e6:.1f} MB")

print()
print("Files in adapter directory:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f"  {f:40s}  {size:.2f} MB")

## 8. Quick Sanity Check

Generate a response right here on Colab before downloading anything.

In [ ]:
model.eval()

test_prompts = [
    "What is the capital of France?",
    "Explain gravity in one sentence.",
    "Write a haiku about autumn leaves.",
]

for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.3,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_toks, skip_special_tokens=True)

    print(f"Q: {prompt}")
    print(f"A: {response}")
    print()

## Next Steps

1. **Download your adapter**: in Colab Files panel, right-click `outputs/adapter/` → Download
2. Place it at `outputs/adapter/` in this repo on your local machine
3. Open [`04_compare.ipynb`](04_compare.ipynb) to compare base vs fine-tuned
4. Open [`05_push_to_hub.ipynb`](05_push_to_hub.ipynb) to publish to HF Hub

---

**Want more?** Natural next experiments:
- Increase `r` from 8 to 16 or 32 — more expressive adapter
- Train for 3 epochs instead of 1
- Add `k_proj` and `o_proj` to `target_modules`
- Try the full 7B SmolLM2 with 4-bit quantization (QLoRA)